# RG Tag Mapping & Enriched AI Cost Table

This notebook creates a reusable RG-to-tag mapping table and builds an enriched AI cost table that fills missing tags using naming conventions.

In [0]:
CATALOG = "automation_s"
SCHEMA = "pierre"
SOURCE_TABLE = "ingestion_azure_p.costs.azure_actual_cost"
MAPPING_TABLE = f"{CATALOG}.{SCHEMA}.rg_tag_mapping"
ENRICHED_TABLE = f"{CATALOG}.{SCHEMA}.ai_foundry_costs_enriched"

print(f"Catalog:       {CATALOG}")
print(f"Schema:        {SCHEMA}")
print(f"Mapping table: {MAPPING_TABLE}")
print(f"Enriched table:{ENRICHED_TABLE}")
print(f"Source table:  {SOURCE_TABLE}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MAPPING_TABLE} (
  rg_pattern STRING,
  inferred_application STRING,
  inferred_environment STRING,
  inferred_company STRING,
  inferred_owner STRING,
  notes STRING
)
COMMENT 'Resource group naming rules used to infer missing Azure tags.'
""")

print(f"Ready: {MAPPING_TABLE}")

In [0]:
spark.sql(f"TRUNCATE TABLE {MAPPING_TABLE}")

spark.sql(f"""
INSERT INTO {MAPPING_TABLE}
VALUES
  ('%MDP-COGNITIVE-T-%', 'MDP Cognitive Services', 'Test', 'Etex Services N.V.', NULL, 'Cognitive services test'),
  ('%MDP-COGNITIVE-P-%', 'MDP Cognitive Services', 'Prod', 'Etex Services N.V.', NULL, 'Cognitive services production'),
  ('%MDP-COGNITIVE-D-%', 'MDP Cognitive Services', 'Dev', 'Etex Services N.V.', NULL, 'Cognitive services development'),

  ('%IA-ETEXMATE-D-%', 'Etex Mate', 'Dev', 'Etex Services N.V.', NULL, 'Etex Mate development'),
  ('%IA-ETEXMATE-P-%', 'Etex Mate', 'Prod', 'Etex Services N.V.', NULL, 'Etex Mate production'),
  ('%IA-ETEXMATE-T-%', 'Etex Mate', 'Test', 'Etex Services N.V.', NULL, 'Etex Mate test'),

  ('%DIGITAL-VAP-D-%', 'VAP', 'Dev', 'Etex Services N.V.', NULL, 'Voice AI Platform'),
  ('%DIGITAL-VAP-P-%', 'VAP', 'Prod', 'Etex Services N.V.', NULL, 'Voice AI Platform'),

  ('%DIGITAL-AISEARCH-D-%', 'AI Search', 'Dev', 'Etex Services N.V.', NULL, 'AI Search development'),
  ('%LOWCODE-COPILOT-D-%', 'Low Code Copilot', 'Dev', 'Etex Services N.V.', NULL, 'Copilot development'),
  ('%LOWCODE-COPILOT-P-%', 'Low Code Copilot', 'Prod', 'Etex Services N.V.', NULL, 'Copilot production'),
  ('%LOWCODE-COPILOT-T-%', 'Low Code Copilot', 'Test', 'Etex Services N.V.', NULL, 'Copilot test'),

  ('%COGNIZANT-GENAIBOT-D-%', 'Cognizant GenAI Bot', 'Dev', 'Etex Services N.V.', NULL, 'Cognizant GenAI Bot development'),
  ('%COGNIZANT-GENAIBOT-P-%', 'Cognizant GenAI Bot', 'Prod', 'Etex Services N.V.', NULL, 'Cognizant GenAI Bot production'),
  ('%COGNIZANT-GENAIBOT-T-%', 'Cognizant GenAI Bot', 'Test', 'Etex Services N.V.', NULL, 'Cognizant GenAI Bot test'),

  ('%COGNIZANT-VOICEAI-D-%', 'Cognizant Voice AI', 'Dev', 'Etex Services N.V.', NULL, 'Cognizant Voice AI development'),
  ('%COGNIZANT-VOICEAI-P-%', 'Cognizant Voice AI', 'Prod', 'Etex Services N.V.', NULL, 'Cognizant Voice AI production'),
  ('%COGNIZANT-VOICEAI-T-%', 'Cognizant Voice AI', 'Test', 'Etex Services N.V.', NULL, 'Cognizant Voice AI test'),

  ('%UIPATH-D-%', 'UiPath', 'Dev', 'Etex Services N.V.', NULL, 'UiPath development'),
  ('%UIPATH-P-%', 'UiPath', 'Prod', 'Etex Services N.V.', NULL, 'UiPath production'),

  ('%hub-citadel-p-%', 'Citadel', 'Prod', 'Etex Services N.V.', NULL, 'Citadel production'),
  ('rg-koen-playground', 'Sandbox', 'Dev', 'Etex Services N.V.', NULL, 'Personal playground')
""")

print(f"Mappings refreshed in {MAPPING_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {ENRICHED_TABLE} AS
WITH source AS (
  SELECT
    a.*,
    UPPER(COALESCE(a.resourcegroupname, '')) AS rg_upper,
    LOWER(CONCAT_WS(' ',
      COALESCE(a.subscriptionname, ''),
      COALESCE(a.resourcegroupname, ''),
      COALESCE(a.consumedservice, ''),
      COALESCE(a.servicefamily, ''),
      COALESCE(a.metercategory, ''),
      COALESCE(a.metersubcategory, ''),
      COALESCE(a.metername, ''),
      COALESCE(a.productname, ''),
      COALESCE(a.additionalinfo, ''),
      COALESCE(a.serviceinfo1, ''),
      COALESCE(a.serviceinfo2, ''),
      COALESCE(a.tags, '')
    )) AS search_text,

    CASE 
      WHEN NULLIF(TRIM(get_json_object(a.tags, '$.Application')), '') = 'EtexMate' THEN 'Etex Mate'
      WHEN NULLIF(TRIM(get_json_object(a.tags, '$.Application')), '') IN ('GenAI-Chatbot', 'GenAi-ChatBot', 'GenAi-Chatbot') THEN 'GenAI-ChatBot'
      WHEN NULLIF(TRIM(get_json_object(a.tags, '$.Application')), '') IN ('Uipath', 'UiPAth') THEN 'UiPath'
      ELSE NULLIF(TRIM(get_json_object(a.tags, '$.Application')), '')
    END AS raw_application,
    NULLIF(TRIM(get_json_object(a.tags, '$.Service')), '') AS raw_service,
    CASE
      WHEN LOWER(NULLIF(TRIM(get_json_object(a.tags, '$.Domain')), '')) = 'finance' THEN 'Finance'
      ELSE NULLIF(TRIM(get_json_object(a.tags, '$.Domain')), '')
    END AS raw_domain,
    NULLIF(TRIM(get_json_object(a.tags, '$.deployment')), '') AS raw_deployment,
    NULLIF(TRIM(get_json_object(a.tags, '$.EnvironmentType')), '') AS raw_environment,
    NULLIF(TRIM(get_json_object(a.tags, '$.Company')), '') AS raw_company,
    NULLIF(TRIM(get_json_object(a.tags, '$.BusinessOwner')), '') AS raw_business_owner
  FROM {SOURCE_TABLE} a
),

mapped AS (
  SELECT
    s.*,
    m.rg_pattern,
    m.inferred_application,
    m.inferred_environment,
    m.inferred_company,
    m.inferred_owner
  FROM source s
  LEFT JOIN {MAPPING_TABLE} m
    ON s.rg_upper LIKE UPPER(m.rg_pattern)
    OR UPPER(s.rg_upper) = UPPER(m.rg_pattern)
),

derived AS (
  SELECT
    m.*,
    CASE
      WHEN search_text RLIKE 'github copilot|copilot studio|microsoft 365 copilot' THEN 'Copilot'
      WHEN search_text RLIKE 'qna maker|azure openai|foundry models|foundry tools|azure translator|defender for ai services' THEN 'AI Foundry / Azure AI'
      WHEN search_text RLIKE 'claude|anthropic|llama|mistral|gpt[- ]|embedding|inference' THEN 'Other model access'
      ELSE 'Non-AI or unclassified'
    END AS api_access,

    COALESCE(
      raw_application,
      inferred_application,
      CASE
        WHEN rg_upper RLIKE 'IA-ETEXMATE' THEN 'Etex Mate'
        WHEN rg_upper RLIKE 'COGNIZANT-GENAIBOT' THEN 'Cognizant GenAI Bot'
        WHEN rg_upper RLIKE 'LOWCODE-COPILOT' THEN 'Low Code Copilot'
        WHEN rg_upper RLIKE 'DIGITAL-AISEARCH' THEN 'AI Search'
        WHEN rg_upper RLIKE 'DIGITAL-VAP' THEN 'VAP'
        WHEN search_text RLIKE 'github copilot' THEN 'GitHub Copilot'
        WHEN search_text RLIKE 'copilot studio' THEN 'Copilot Studio'
        ELSE NULL
      END
    ) AS application,

    COALESCE(
      raw_service,
      CASE
        WHEN NULLIF(TRIM(consumedservice), '') IS NOT NULL THEN
          CASE UPPER(TRIM(consumedservice))
            WHEN 'MICROSOFT.APP' THEN 'Microsoft.App'
            WHEN 'MICROSOFT.CDN' THEN 'Microsoft.Cdn'
            WHEN 'MICROSOFT.COMPUTE' THEN 'Microsoft.Compute'
            WHEN 'MICROSOFT.CONTAINERREGISTRY' THEN 'Microsoft.ContainerRegistry'
            WHEN 'MICROSOFT.DATAFACTORY' THEN 'Microsoft.DataFactory'
            WHEN 'MICROSOFT.DBFORMYSQL' THEN 'Microsoft.DBforMySQL'
            WHEN 'MICROSOFT.DOCUMENTDB' THEN 'Microsoft.DocumentDb'
            WHEN 'MICROSOFT.HYBRIDCOMPUTE' THEN 'Microsoft.HybridCompute'
            WHEN 'MICROSOFT.INSIGHTS' THEN 'Microsoft.Insights'
            WHEN 'MICROSOFT.KEYVAULT' THEN 'Microsoft.KeyVault'
            WHEN 'MICROSOFT.MONITOR' THEN 'Microsoft.Monitor'
            WHEN 'MICROSOFT.NETWORK' THEN 'Microsoft.Network'
            WHEN 'MICROSOFT.SERVICEBUS' THEN 'Microsoft.ServiceBus'
            WHEN 'MICROSOFT.STORAGE' THEN 'Microsoft.Storage'
            WHEN 'MICROSOFT.SYNAPSE' THEN 'Microsoft.Synapse'
            WHEN 'MICROSOFT.WEB' THEN 'Microsoft.Web'
            ELSE TRIM(consumedservice)
          END
        ELSE NULL
      END,
      NULLIF(TRIM(servicefamily), ''),
      NULLIF(TRIM(metercategory), ''),
      'Unknown'
    ) AS service,

    COALESCE(
      raw_deployment,
      CASE
        WHEN metername RLIKE '(?i)gpt[- ]?[0-9]|claude|anthropic|llama|mistral' THEN metername
        WHEN productname RLIKE '(?i)gpt[- ]?[0-9]|claude|anthropic|llama|mistral' THEN productname
        WHEN search_text RLIKE '(?i)github copilot' THEN 'GitHub Copilot'
        WHEN search_text RLIKE '(?i)copilot studio' THEN 'Copilot Studio'
        ELSE NULL
      END
    ) AS model,

    COALESCE(
      raw_domain,
      CASE
        WHEN search_text RLIKE 'github copilot|copilot studio' THEN 'Intelligent Automation'
        WHEN rg_upper RLIKE 'SPOKE-IA' THEN 'Intelligent Automation'
        WHEN rg_upper RLIKE 'SPOKE-MDP' THEN 'Modern Data Platform'
        WHEN rg_upper RLIKE 'SPOKE-SAP' THEN 'SAP'
        WHEN rg_upper RLIKE 'SPOKE-DIGITAL' THEN 'Digital'
        WHEN rg_upper RLIKE 'SPOKE-LOWCODE' THEN 'Low Code'
        WHEN rg_upper RLIKE 'SPOKE-DMZ' THEN 'DMZ'
        WHEN rg_upper RLIKE 'SPOKE-DATA' THEN 'Secure Data'
        WHEN rg_upper RLIKE 'SPOKE-URSA' THEN 'URSA'
        WHEN rg_upper RLIKE 'SPOKE-APPS' THEN 'Apps'
        WHEN rg_upper RLIKE 'SPOKE-NEWWAYS' THEN 'New Ways'
        WHEN rg_upper RLIKE 'SPOKE-PLADUR' THEN 'Pladur Ecommerce'
        WHEN rg_upper RLIKE 'HUB-' THEN 'Shared Services'
        WHEN raw_domain RLIKE '(?i)^shared$|^shared services$' THEN 'Shared Services'
        WHEN raw_domain RLIKE '(?i)^foundation(s)?$' THEN 'Foundations'
        WHEN raw_domain RLIKE '(?i)^central functions$' THEN 'Central Functions'
        ELSE 'Unassigned'
      END
    ) AS spoke,

    CASE
      WHEN search_text RLIKE 'defender for ai services' THEN 'Defender for AI Services'
      WHEN metercategory RLIKE '(?i)azure openai' OR metersubcategory RLIKE '(?i)azure openai' OR metername RLIKE '(?i)azure openai' OR productname RLIKE '(?i)azure openai' THEN 'Azure OpenAI'
      WHEN search_text RLIKE 'azure translator' THEN 'Azure Translator'
      WHEN search_text RLIKE 'azure speech|voice live' THEN 'Azure Speech'
      WHEN search_text RLIKE 'azure document intelligence' THEN 'Azure Document Intelligence'
      WHEN search_text RLIKE 'azure custom vision' THEN 'Azure Custom Vision'
      WHEN search_text RLIKE 'azure vision' THEN 'Azure Vision'
      WHEN search_text RLIKE 'azure language' THEN 'Azure Language'
      WHEN search_text RLIKE 'qna maker' THEN 'QnA Maker'
      WHEN search_text RLIKE 'github copilot' THEN 'GitHub Copilot'
      WHEN search_text RLIKE 'copilot studio' THEN 'Copilot Studio'
      WHEN search_text RLIKE 'claude|anthropic' THEN 'Claude / Anthropic'
      WHEN search_text RLIKE 'llama' THEN 'Llama'
      WHEN search_text RLIKE 'mistral' THEN 'Mistral'
      ELSE NULL
    END AS ai_capability

  FROM mapped m
)

SELECT
  CAST(date AS DATE) AS cost_date,
  subscriptionname,
  subscriptionid,
  UPPER(resourcegroupname) AS resourcegroupname,
  resourceid,
  consumedservice,
  servicefamily,
  metercategory,
  metersubcategory,
  metername,
  productname,
  pricingcurrency,
  billingcurrency,
  costinpricingcurrency,
  costinbillingcurrency,
  CASE
    WHEN UPPER(billingcurrency) = 'EUR' THEN costinbillingcurrency
    WHEN UPPER(pricingcurrency) = 'EUR' THEN costinpricingcurrency
    ELSE NULL
  END AS cost_eur,
  costinusd AS cost_usd,
  paygcostinbillingcurrency,
  paygcostinusd,
  quantity,
  tags,
  application,
  service,
  model,
  spoke,
  api_access,
  ai_capability,
  COALESCE(inferred_environment, raw_environment) AS environment,
  COALESCE(inferred_company, raw_company) AS company,
  CASE
    WHEN rg_pattern IS NOT NULL AND raw_application IS NULL THEN TRUE
    ELSE FALSE
  END AS application_inferred,
  CASE
    WHEN rg_pattern IS NOT NULL AND raw_domain IS NULL THEN TRUE
    ELSE FALSE
  END AS spoke_inferred,
  CASE
    WHEN model IS NOT NULL AND raw_deployment IS NULL THEN TRUE
    ELSE FALSE
  END AS model_inferred,
  batch_id,
  load_ts
FROM derived
""")

row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {ENRICHED_TABLE}").collect()[0]["row_count"]
print(f"Created {ENRICHED_TABLE} with {row_count:,} rows")

In [0]:
display(spark.sql(f"SELECT * FROM {MAPPING_TABLE} ORDER BY inferred_application, inferred_environment"))

In [0]:
display(spark.sql(f"""
SELECT
  resourcegroupname,
  application,
  service,
  model,
  spoke,
  api_access,
  SUM(cost_eur) AS cost_eur,
  SUM(cost_usd) AS cost_usd,
  COUNT(*) AS row_count
FROM {ENRICHED_TABLE}
WHERE application IS NULL OR model IS NULL OR spoke IS NULL
GROUP BY resourcegroupname, application, service, model, spoke, api_access
ORDER BY cost_eur DESC
"""))

In [0]:
display(spark.sql(f"""
SELECT
  api_access,
  spoke,
  application,
  service,
  model,
  COUNT(*) AS row_count,
  ROUND(SUM(cost_eur), 2) AS cost_eur,
  ROUND(SUM(cost_usd), 2) AS cost_usd
FROM {ENRICHED_TABLE}
GROUP BY api_access, spoke, application, service, model
ORDER BY cost_eur DESC
"""))

## Quick use

1. Set the catalog and schema in the config cell.
2. Run all cells.
3. Validate the mapping and enriched cost output.